# DiskANN Plotting

DiskANN/FreshDiskANN benchmark log를 읽어서 표와 그래프로 확인하는 노트북입니다.

- `search_memory_index`
- `search_disk_index`
- `motivation`
- `overall_performance`

같은 공백 정렬 로그를 바로 읽을 수 있게 만들었습니다.

예시 로그 저장:

```bash
mkdir -p logs
build/tests/search_disk_index ... | tee logs/search_disk_index.log
build/tests/search_memory_index ... | tee logs/search_memory_index.log
```


In [ ]:
from pathlib import Path

from plotting_utils import load_many_logs, maybe_to_dataframe, write_csv

# 여기에 분석할 로그 경로를 넣어 주세요.
LOG_PATHS = [
    Path("logs/search_disk_index.log"),
    # Path("logs/search_memory_index.log"),
    # Path("logs/motivation.log"),
]

if not LOG_PATHS:
    raise RuntimeError("LOG_PATHS에 로그 파일 경로를 하나 이상 넣어 주세요.")

missing_paths = [path for path in LOG_PATHS if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "로그 파일을 찾을 수 없습니다: " + ", ".join(str(path) for path in missing_paths)
    )

rows = load_many_logs(LOG_PATHS)
if not rows:
    raise RuntimeError("로그는 읽었지만 파싱된 행이 없습니다. 로그 형식이나 파일 내용을 확인해 주세요.")

table = maybe_to_dataframe(rows)

print(f"parsed rows: {len(rows)}")
table.head() if hasattr(table, "head") else table[:5]


In [ ]:
# 필요하면 CSV로 내보낼 수 있습니다.
csv_path = write_csv(rows, Path("parsed_benchmark_rows.csv"))
csv_path


In [ ]:
if not hasattr(table, "columns"):
    raise RuntimeError(
        "DataFrame 미리보기를 위해 pandas가 필요합니다. `python3 -m pip install pandas matplotlib jupyter`를 실행해 주세요."
    )

numeric_columns = table.select_dtypes(include=["number"]).columns.tolist()
print("numeric columns:", numeric_columns)
table


In [ ]:
import importlib.util

if importlib.util.find_spec("matplotlib") is None:
    raise RuntimeError(
        "matplotlib가 필요합니다. `python3 -m pip install matplotlib pandas jupyter`를 실행해 주세요."
    )

import matplotlib.pyplot as plt

df = table.copy()

x_col = "recall" if "recall" in df.columns and df["recall"].notna().any() else "l_search"
latency_col = "p999_latency_ms" if "p999_latency_ms" in df.columns and df["p999_latency_ms"].notna().any() else "mean_latency_ms"
group_col = "iteration" if "iteration" in df.columns and df["iteration"].notna().any() else "source"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for group_value, part in df.groupby(group_col, dropna=False):
    part = part.sort_values(x_col)
    label = f"{group_col}={group_value}"
    axes[0].plot(part[x_col], part[latency_col], marker="o", linewidth=2, label=label)

axes[0].set_title(f"{latency_col} vs {x_col}")
axes[0].set_xlabel(x_col)
axes[0].set_ylabel(latency_col)
axes[0].grid(alpha=0.3)
axes[0].legend()

if "qps" in df.columns and df["qps"].notna().any():
    for group_value, part in df.groupby(group_col, dropna=False):
        label = f"{group_col}={group_value}"
        axes[1].scatter(part[x_col], part["qps"], s=60, label=label)
    axes[1].set_title(f"qps vs {x_col}")
    axes[1].set_xlabel(x_col)
    axes[1].set_ylabel("qps")
    axes[1].grid(alpha=0.3)
    axes[1].legend()
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# 필요하면 필터링해서 다시 그릴 수 있습니다.
# 예: 특정 iteration만 보기
# subset = df[df["iteration"] == 0].copy()
# subset.sort_values("l_search")

# 예: figure 저장
# fig.savefig("benchmark_plot.png", dpi=200, bbox_inches="tight")
